# Andina Market · ETL en PySpark + Delta (Fabric o Databricks)

Mismo proceso que `python -m andina_etl`, escrito para Spark: **CSV → Bronze (Parquet crudo) → Silver (Delta, MERGE idempotente) → cuarentena → venta neta diaria (Delta)**.

**Cómo ejecutarlo**

- **Fabric:** crea un Lakehouse, sube los 4 CSV a `Files/andina/raw/`, adjunta el Lakehouse al notebook y ejecuta todas las celdas.
- **Databricks:** crea el catálogo/esquema `andina.datos` y un Volume con los CSV, cambia `PLATAFORMA` en la celda 1 y ejecuta todas las celdas.
- **Idempotencia:** vuelve a ejecutar todo. La celda de resumen debe mostrar 0 insertados y 0 actualizados.
- **Verificación:** la última celda compara contra las cifras del ETL local (19.270 pedidos y 20.124 pagos válidos; venta neta 14.177.333.600).

**Qué está probado y qué no**

- Probado con PySpark 3.5 local y ANSI activado: las reglas de calidad (celda 2) dan **los mismos conteos por regla, los mismos válidos y la misma venta neta** que la versión pandas/DuckDB sobre los datos reales.
- **No probado:** `MERGE INTO`, `DELETE` y `DESCRIBE HISTORY` de Delta, las rutas de Fabric/Databricks y la lectura `binaryFile`. Se escribieron con la sintaxis estándar de Delta, pero no se ejecutaron en ninguna plataforma. Si algo falla, probablemente sea ahí.
- **Caídas a mitad de corrida:** Delta no tiene transacciones entre tablas (la versión DuckDB sí usa una). Si el notebook falla después de escribir Silver y antes de Gold, vuelve a ejecutarlo con `RECALCULAR_GOLD = True` en la celda 1. Lo mismo tras cambiar la definición de venta.


In [ ]:
# === 1. Configuración: lo único que cambia entre Fabric y Databricks ===
import uuid
import hashlib
from datetime import datetime
from pyspark.sql import functions as F

PLATAFORMA = "fabric"            # "fabric" o "databricks"
FECHA_MIN, FECHA_MAX = "2025-01-01", "2026-09-30"   # rango válido de fecha_pedido
LOOKBACK_DIAS = 3                # ventana de reproceso de pagos sobre fecha_carga
ESTADO_VENTA = "ENTREGADO"       # lo único que cuenta como venta
RECALCULAR_GOLD = False          # True: recalcula todos los días (tras una caída o un cambio de definición)

if PLATAFORMA == "fabric":
    # Lakehouse por defecto adjunto al notebook
    RAW, BRONZE = "Files/andina/raw", "Files/andina/bronze"
    def T(nombre):               # tablas del Lakehouse: silver_pedidos, cuarentena, ...
        return nombre
else:
    # Unity Catalog + Volumes
    RAW, BRONZE = "/Volumes/andina/datos/raw", "/Volumes/andina/datos/bronze"
    def T(nombre):
        return f"andina.datos.{nombre}"


In [ ]:
# === 2. Reglas de calidad (espejo de la versión pandas; ver spark/rules_spark.py) ===
"""Reglas de calidad en PySpark. Espejo de andina_etl/rules.py (versión pandas/DuckDB).

Mismos códigos de regla, mismo criterio de duplicados, mismo hash de fila sobre los datos
crudos. Sirve para correr en Fabric o Databricks. No depende de Delta: son funciones puras
de DataFrame, por eso se pueden probar con PySpark local.

Todo el parseo usa try_cast / try_to_timestamp: con ANSI activado (por defecto en algunos
runtimes de Databricks) un valor sucio produciría una excepción en vez de NULL.
"""
from __future__ import annotations

from functools import reduce

from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F

CANALES = {
    "WEB": "WEB", "PAGINA WEB": "WEB",
    "APP": "APP", "APLICACION": "APP",
    "TIENDA": "TIENDA", "TIENDA FISICA": "TIENDA",
}
ESTADOS_PEDIDO = ["ENTREGADO", "DEVUELTO", "CANCELADO"]
MEDIOS_PAGO = ["Tarjeta crédito", "PSE", "Billetera digital", "Contraentrega"]

COLS_CLIENTES = ["id_cliente", "nombre", "documento", "ciudad", "canal_registro", "fecha_registro"]
COLS_PEDIDOS = ["id_pedido", "id_cliente", "fecha_pedido", "canal", "categoria",
                "valor_bruto", "descuento", "estado", "fecha_entrega"]
COLS_PAGOS = ["id_pago", "id_pedido", "fecha_pago", "valor", "medio_pago", "fecha_carga"]


# ---------------------------------------------------------------- utilidades

def _txt(nombre: str) -> Column:
    """Texto sin espacios laterales; vacío -> NULL."""
    limpio = F.trim(F.col(nombre))
    return F.when(limpio == "", F.lit(None)).otherwise(limpio)


def _pesos(nombre: str) -> Column:
    # Mismo contrato de pesos enteros que pandas. Sin paso por double.
    texto = F.trim(F.col(nombre))
    return F.when(texto.rlike(r"^[+-]?\d+(\.0+)?$"),
                  F.expr(f"try_cast(try_cast({nombre} as decimal(38,0)) as bigint)"))


def _ts(nombre: str) -> Column:
    """Fecha o fecha-hora ISO; NULL si no se puede leer."""
    return F.coalesce(
        F.expr(f"try_to_timestamp({nombre}, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr(f"try_to_timestamp({nombre}, 'yyyy-MM-dd')"),
    )


def canal_normalizado(nombre: str) -> Column:
    """'app ', 'APLICACION', 'Pagina Web' -> WEB / APP / TIENDA; NULL si no está en el mapa."""
    k = F.upper(F.trim(F.regexp_replace(
        F.translate(F.col(nombre), "ÁÉÍÓÚÜáéíóúü", "AEIOUUaeiouu"), r"\s+", " ")))
    expr = None
    for destino in ("WEB", "APP", "TIENDA"):
        cond = k.isin([c for c, d in CANALES.items() if d == destino])
        expr = F.when(cond, destino) if expr is None else expr.when(cond, destino)
    return expr


def hash_cols(columnas: list[str]) -> Column:
    """sha256 de las columnas crudas unidas con '|' (NULL = ''). Idéntico al de la versión pandas."""
    return F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in columnas]), 256)


def documento_limpio(nombre: str) -> Column:
    solo = F.regexp_replace(F.regexp_replace(F.col(nombre), r"\D", ""), r"^0+", "")
    return F.when(solo == "", F.lit(None)).otherwise(solo)


def _alguna(reglas: dict[str, Column]) -> Column:
    return reduce(lambda a, b: a | b, [F.coalesce(c, F.lit(False)) for c in reglas.values()])


def _cuarentena(df: DataFrame, tabla: str, cols_origen: list[str], reglas: dict[str, Column]) -> DataFrame:
    """Una fila por (registro, regla violada), con la fila original completa en JSON."""
    col_id = cols_origen[0]
    partes = [
        df.filter(F.coalesce(cond, F.lit(False))).select(
            F.lit(tabla).alias("tabla_origen"),
            F.coalesce(F.col(col_id), F.lit("")).alias("id_registro"),
            F.lit(codigo).alias("regla_violada"),
            F.col("row_hash"),
            F.to_json(F.struct(*cols_origen), {"ignoreNullFields": "false"}).alias("registro_json"),
        )
        for codigo, cond in reglas.items()
    ]
    return reduce(DataFrame.unionByName, partes)


def _con_veces_id(df: DataFrame, col_id: str) -> DataFrame:
    """Agrega x_veces_id: cuántas veces aparece el id en el lote. Va como columna porque Spark
    no admite funciones de ventana dentro de un filtro."""
    return df.withColumn("x_veces_id", F.count(F.lit(1)).over(Window.partitionBy(col_id)))


def _id_repetido(col_id: str) -> Column:
    """True si el id aparece más de una vez en el lote: todas las copias van a cuarentena."""
    return F.col(col_id).isNotNull() & (F.col("x_veces_id") > 1)


def _desempatar_duplicados(cand: DataFrame, col_id: str, existente: DataFrame) -> DataFrame:
    """Marca x_dup. Gana el que ya está en Silver; si ninguno está, el de menor id."""
    ganador_lote = F.min(col_id).over(Window.partitionBy("hash_negocio"))
    ex = existente.select("hash_negocio", F.col(existente.columns[1]).alias("x_ganador_silver"))
    return (cand.withColumn("x_ganador_lote", ganador_lote)
            .join(ex, "hash_negocio", "left")
            .withColumn("x_dup", F.col(col_id) != F.coalesce(F.col("x_ganador_silver"), F.col("x_ganador_lote"))))


# ------------------------------------------------------------------- clientes

def validar_clientes(raw: DataFrame) -> tuple[DataFrame, DataFrame]:
    df = (raw.withColumn("row_hash", hash_cols(COLS_CLIENTES))
          .withColumn("x_id", _txt("id_cliente"))
          .withColumn("x_fecha", F.expr("try_to_timestamp(fecha_registro, 'yyyy-MM-dd')")))
    n = F.row_number().over(Window.partitionBy("x_id").orderBy("row_hash"))
    df = df.withColumn("x_n", F.when(F.col("x_id").isNotNull(), n))
    reglas = {
        "ID_CLIENTE_NULO": F.col("x_id").isNull(),
        "FECHA_REGISTRO_INVALIDA": F.col("x_fecha").isNull(),
        "ID_CLIENTE_DUPLICADO": F.col("x_n") > 1,
    }
    cuarentena = _cuarentena(df, "clientes", COLS_CLIENTES, reglas)
    ok = df.filter(~_alguna(reglas))
    out = ok.select(
        F.col("x_id").alias("id_cliente"),
        _txt("nombre").alias("nombre"),
        F.col("documento").alias("documento_original"),
        documento_limpio("documento").alias("documento_limpio"),
        _txt("ciudad").alias("ciudad"),
        canal_normalizado("canal_registro").alias("canal_registro"),
        F.col("x_fecha").cast("date").alias("fecha_registro"),
        F.trunc(F.col("x_fecha"), "month").alias("cohorte_registro"),
        F.col("row_hash"),
    )
    return out, cuarentena


# -------------------------------------------------------------------- pedidos

def validar_pedidos(raw: DataFrame, fecha_min: str, fecha_max: str,
                    clientes_ids: DataFrame, negocio_existente: DataFrame) -> tuple[DataFrame, DataFrame]:
    """clientes_ids: DataFrame(id_cliente). negocio_existente: DataFrame(hash_negocio, id_pedido),
    una fila por hash (el id menor ya cargado en Silver)."""
    df = (raw
          .withColumn("row_hash", hash_cols(COLS_PEDIDOS))
          .withColumn("hash_negocio", hash_cols(["id_cliente", "fecha_pedido", "valor_bruto", "categoria"]))
          .withColumn("x_id", _txt("id_pedido"))
          .withColumn("x_cli", _txt("id_cliente"))
          .withColumn("x_canal", canal_normalizado("canal"))
          .withColumn("x_estado", F.upper(_txt("estado")))
          .withColumn("x_fp", _ts("fecha_pedido"))
          .withColumn("x_fe", _ts("fecha_entrega"))
          .withColumn("x_vb", _pesos("valor_bruto"))
          .withColumn("x_ds", _pesos("descuento"))
          .withColumn("x_dia", F.date_trunc("day", F.col("x_fp"))))
    df = _con_veces_id(df, "x_id")

    fmin, fmax = F.lit(fecha_min).cast("timestamp"), F.lit(fecha_max).cast("timestamp")
    c = F.col
    reglas = {
        "ID_PEDIDO_NULO": c("x_id").isNull(),
        "ID_PEDIDO_DUPLICADO": _id_repetido("x_id"),
        "CANAL_DESCONOCIDO": c("x_canal").isNull(),
        "ESTADO_INVALIDO": ~F.coalesce(c("x_estado").isin(ESTADOS_PEDIDO), F.lit(False)),
        "FECHA_PEDIDO_INVALIDA": c("x_fp").isNull(),
        "FECHA_FUERA_DE_RANGO": c("x_fp").isNotNull() & ((c("x_dia") < fmin) | (c("x_dia") > fmax)),
        "VALOR_BRUTO_NO_POSITIVO": c("x_vb").isNull() | (c("x_vb") <= 0),
        "DESCUENTO_INVALIDO": c("x_ds").isNull() | (c("x_ds") < 0) | (c("x_ds") > c("x_vb")),
        "ENTREGADO_SIN_FECHA": (c("x_estado") == "ENTREGADO") & c("x_fe").isNull(),
        "ENTREGA_ANTERIOR_A_PEDIDO": c("x_fe").isNotNull() & c("x_fp").isNotNull() & (c("x_fe") < c("x_dia")),
    }
    df = df.withColumn("x_malo", _alguna(reglas))
    cand = _desempatar_duplicados(df.filter(~c("x_malo")), "x_id", negocio_existente)

    cuarentena = _cuarentena(df, "pedidos", COLS_PEDIDOS, reglas).unionByName(
        _cuarentena(cand, "pedidos", COLS_PEDIDOS, {"DUPLICADO_LOGICO": c("x_dup")}))

    conocidos = clientes_ids.select(c("id_cliente").alias("x_conocido"), F.lit(True).alias("x_ok"))
    valido = (cand.filter(~c("x_dup"))
              .join(conocidos, c("x_cli") == c("x_conocido"), "left"))
    out = valido.select(
        c("x_id").alias("id_pedido"),
        c("x_cli").alias("id_cliente"),
        c("x_fp").alias("fecha_pedido"),
        c("x_dia").cast("date").alias("fecha_pedido_dia"),
        c("x_canal").alias("canal"),
        _txt("categoria").alias("categoria"),
        c("x_vb").cast("long").alias("valor_bruto"),
        c("x_ds").cast("long").alias("descuento"),
        (c("x_vb") - c("x_ds")).cast("long").alias("valor_neto"),
        c("x_estado").alias("estado"),
        c("x_fe").cast("date").alias("fecha_entrega"),
        c("x_ok").isNull().alias("flag_cliente_desconocido"),
        c("hash_negocio"),
        c("row_hash"),
    )
    return out, cuarentena


# ---------------------------------------------------------------------- pagos

def _fecha_pago() -> Column:
    """yyyy-MM-dd y dd/MM/yyyy con formato explícito: '03/04/2026' es 3 de abril."""
    return F.coalesce(
        F.when(F.col("fecha_pago").rlike(r"^\d{4}-\d{2}-\d{2}$"),
               F.expr("try_to_timestamp(fecha_pago, 'yyyy-MM-dd')")),
        F.when(F.col("fecha_pago").rlike(r"^\d{2}/\d{2}/\d{4}$"),
               F.expr("try_to_timestamp(fecha_pago, 'dd/MM/yyyy')")),
    )


def validar_pagos(raw: DataFrame, pedidos_dia: DataFrame, pedidos_en_cuarentena: DataFrame,
                  negocio_existente: DataFrame) -> tuple[DataFrame, DataFrame]:
    """pedidos_dia: DataFrame(id_pedido, fecha_pedido_dia) de Silver, un id por fila.
    pedidos_en_cuarentena: DataFrame(id_registro) con los ids de pedido rechazados."""
    c = F.col
    pd_ = pedidos_dia.select(c("id_pedido").alias("x_ped"), c("fecha_pedido_dia").cast("timestamp").alias("x_fped"))
    pq = pedidos_en_cuarentena.select(c("id_registro").alias("x_q"), F.lit(True).alias("x_en_q"))
    df = (raw
          .withColumn("row_hash", hash_cols(COLS_PAGOS))
          .withColumn("x_id", _txt("id_pago"))
          .withColumn("x_pedido", _txt("id_pedido"))
          .withColumn("x_fpago", _fecha_pago())
          .withColumn("x_fcarga", F.expr("try_to_timestamp(fecha_carga, 'yyyy-MM-dd')"))
          .withColumn("x_val", _pesos("valor"))
          .withColumn("x_medio", _txt("medio_pago"))
          .join(pd_, c("x_pedido") == c("x_ped"), "left")
          .join(pq, c("x_pedido") == c("x_q"), "left"))
    df = _con_veces_id(df, "x_id")
    en_silver = c("x_fped").isNotNull()
    en_q = F.coalesce(c("x_en_q"), F.lit(False))
    reglas = {
        "ID_PAGO_NULO": c("x_id").isNull(),
        "ID_PAGO_DUPLICADO": _id_repetido("x_id"),
        "ID_PEDIDO_NULO": c("x_pedido").isNull(),
        "FECHA_PAGO_INVALIDA": c("x_fpago").isNull(),
        "FECHA_CARGA_INVALIDA": c("x_fcarga").isNull(),
        "VALOR_INVALIDO": c("x_val").isNull() | (c("x_val") == 0),
        "MEDIO_PAGO_INVALIDO": ~F.coalesce(c("x_medio").isin(MEDIOS_PAGO), F.lit(False)),
        "PEDIDO_EN_CUARENTENA": c("x_pedido").isNotNull() & ~en_silver & en_q,
        "PAGO_HUERFANO": c("x_pedido").isNotNull() & ~en_silver & ~en_q,
        "PAGO_ANTERIOR_A_PEDIDO": en_silver & c("x_fpago").isNotNull() & (c("x_fpago") < c("x_fped")),
    }
    df = (df.withColumn("x_malo", _alguna(reglas))
          .withColumn("hash_negocio", F.sha2(F.concat_ws(
              "|", F.coalesce(c("x_pedido"), F.lit("")),
              F.coalesce(F.date_format(c("x_fpago"), "yyyy-MM-dd"), F.lit("")),
              F.coalesce(c("x_val").cast("long").cast("string"), F.lit("")),
              F.coalesce(c("x_medio"), F.lit(""))), 256)))
    cand = _desempatar_duplicados(df.filter(~c("x_malo")), "x_id", negocio_existente)

    cuarentena = _cuarentena(df, "pagos", COLS_PAGOS, reglas).unionByName(
        _cuarentena(cand, "pagos", COLS_PAGOS, {"DUPLICADO_LOGICO": c("x_dup")}))
    out = cand.filter(~c("x_dup")).select(
        c("x_id").alias("id_pago"),
        c("x_pedido").alias("id_pedido"),
        c("x_fpago").cast("date").alias("fecha_pago"),
        c("x_val").cast("long").alias("valor"),
        (c("x_val") < 0).alias("es_reembolso"),
        c("x_medio").alias("medio_pago"),
        c("x_fcarga").cast("date").alias("fecha_carga"),
        c("hash_negocio"),
        c("row_hash"),
    )
    return out, cuarentena


In [ ]:
# === 3. Extensiones: soporte, estrella, contratos, incidencias y publicación
# (espejo de andina_etl/{tickets,star,contracts,operations}.py; ver spark/extensiones_spark.py) ===
from pyspark.sql.utils import AnalysisException
"""IA-R02: las extensiones locales (soporte, estrella, contratos, incidencias y
publicación consistente) escritas para el flujo Delta de Fabric o Databricks.

Por qué este archivo existe aparte de `rules_spark.py`: aquél son reglas de fila,
funciones puras de DataFrame. Esto es orquestación sobre tablas, y en Delta no hay
transacción entre tablas como en DuckDB. La diferencia obliga a un diseño distinto,
no a una traducción literal:

- **Publicación consistente sin transacción.** Local envuelve toda la corrida en una
  transacción DuckDB: si algo falla, no queda nada a medias. Delta solo garantiza
  atomicidad por tabla. Aquí la consistencia se consigue al revés: los hechos se
  escriben con el `run_id` de la corrida, se concilia contra Silver y solo si
  cuadra se mueve el puntero `publicacion_actual`. Los consumidores leen las vistas
  `v_fact_*`, que filtran por el run publicado, así que nunca ven una carga a medias
  ni una que no pasó las compuertas.
- **Separación por testabilidad.** Las sentencias que no dependen de Delta
  (`INSERT INTO`, `INSERT OVERWRITE`, `SELECT`) se devuelven como texto y se prueban
  contra Spark local comparando con el resultado de DuckDB. Las que sí dependen de
  Delta (`MERGE`, `UPDATE`, `DELETE`) viven en funciones con el prefijo `delta_` y
  **no se han ejecutado**: su validación es E04.

Las claves sustitutas usan el mismo criterio de orden que la versión DuckDB
(`ORDER BY` sobre el valor natural), de modo que ambos motores asignan las mismas
claves sobre los mismos datos y la comparación fila a fila es exacta.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F



COLS_TICKETS = ['id_ticket', 'id_cliente', 'id_pedido', 'fecha_creacion', 'canal',
                'estado', 'fecha_cierre', 'descripcion']
MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto',
         'septiembre', 'octubre', 'noviembre', 'diciembre']


# --------------------------------------------------------------------- soporte

def _fecha_ticket(nombre: str):
    """ISO con o sin hora, igual que `pandas.to_datetime(format='ISO8601')`."""
    return F.coalesce(F.expr(f"try_to_timestamp({nombre}, 'yyyy-MM-dd HH:mm:ss')"),
                      F.expr(f"try_to_timestamp({nombre}, 'yyyy-MM-dd')"))


def hash_texto_normalizado(nombre: str):
    """Espejo exacto de `andina_etl.etiquetas.hash_texto`.

    Mismo orden de operaciones que la versión pandas (traducir acentos, minúsculas,
    colapsar espacios) y el mismo juego de caracteres: si divergieran, las etiquetas
    revisadas dejarían de casar con los tickets en uno de los dos motores.
    """
    sin_acentos = F.translate(F.col(nombre), 'ÁÉÍÓÚÜÑáéíóúüñ', 'AEIOUUNaeiouun')
    return F.sha2(F.trim(F.regexp_replace(F.lower(sin_acentos), r'\s+', ' ')), 256)


def validar_tickets(raw: DataFrame) -> tuple[DataFrame, DataFrame]:
    """Espejo de `andina_etl.tickets.validar_tickets`.

    Las fechas incoherentes no se descartan: el ticket se conserva con su bandera y
    sin horas de resolución, para no inventar un SLA que la fuente no respalda.
    """
    df = (raw.withColumn('row_hash', hash_cols(COLS_TICKETS))
          .withColumn('x_id', _txt('id_ticket'))
          .withColumn('x_creada', _fecha_ticket('fecha_creacion'))
          .withColumn('x_cerrada', _fecha_ticket('fecha_cierre')))
    df = df.withColumn('x_veces', F.count(F.lit(1)).over(Window.partitionBy('x_id')))
    reglas = {
        'ID_TICKET_NULO': F.col('x_id').isNull(),
        # El id repetido manda todas las copias a cuarentena: no se sabe cuál vale.
        'ID_TICKET_DUPLICADO': F.col('x_id').isNotNull() & (F.col('x_veces') > 1),
        'FECHA_CREACION_INVALIDA': F.col('x_creada').isNull(),
    }
    cuarentena = _cuarentena(df, 'tickets_soporte', COLS_TICKETS, reglas)
    malo = (F.coalesce(reglas['ID_TICKET_NULO'], F.lit(False))
            | F.coalesce(reglas['ID_TICKET_DUPLICADO'], F.lit(False))
            | F.coalesce(reglas['FECHA_CREACION_INVALIDA'], F.lit(False)))

    estado = F.upper(_txt('estado'))
    invalido = F.col('x_cerrada').isNotNull() & (F.col('x_cerrada') < F.col('x_creada'))
    sin_fecha = (estado == 'CERRADO') & F.col('x_cerrada').isNull()
    horas = (F.col('x_cerrada').cast('double') - F.col('x_creada').cast('double')) / 3600
    out = df.filter(~malo).select(
        F.col('x_id').alias('id_ticket'),
        _txt('id_cliente').alias('id_cliente'),
        _txt('id_pedido').alias('id_pedido'),
        F.col('x_creada').alias('fecha_creacion'),
        _txt('canal').alias('canal_soporte'),
        estado.alias('estado'),
        F.col('x_cerrada').alias('fecha_cierre'),
        F.col('descripcion').alias('descripcion_original'),
        hash_texto_normalizado('descripcion').alias('texto_hash'),
        F.coalesce(invalido, F.lit(False)).alias('flag_cierre_invalido'),
        F.coalesce(sin_fecha, F.lit(False)).alias('flag_cerrado_sin_fecha'),
        F.when((estado == 'CERRADO') & ~F.coalesce(invalido, F.lit(False))
               & ~F.coalesce(sin_fecha, F.lit(False)), horas).alias('horas_resolucion'),
        F.col('row_hash'),
    )
    return out, cuarentena


# ------------------------------------------------------------ DDL de extensión

DDL_EXTENSIONES = {
    'silver_tickets': """id_ticket STRING, id_cliente STRING, id_pedido STRING,
        fecha_creacion TIMESTAMP, canal_soporte STRING, estado STRING, fecha_cierre TIMESTAMP,
        descripcion_original STRING, texto_hash STRING, flag_cierre_invalido BOOLEAN,
        flag_cerrado_sin_fecha BOOLEAN, horas_resolucion DOUBLE, row_hash STRING,
        ingested_at TIMESTAMP""",
    'calidad_incidencias': """tabla STRING, id_registro STRING, regla STRING, estado STRING,
        primera_observacion TIMESTAMP, ultima_revision TIMESTAMP, run_id STRING, motivo STRING""",
    # tipo + estado + origen: la propuesta de un clasificador y la decisión de una
    # persona se guardan por separado, nunca en la misma columna.
    'ticket_clasificacion': """texto_hash STRING, version STRING, tipo STRING, estado STRING,
        origen STRING, anotadores STRING, motivo STRING, particion STRING, confianza DOUBLE,
        importado_en TIMESTAMP""",
    'publicacion_actual': """id INT, run_id STRING, validado_en TIMESTAMP, venta_neta BIGINT""",
    'gold_dim_fecha': """fecha_key INT, fecha DATE, anio SMALLINT, trimestre SMALLINT, mes SMALLINT,
        nombre_mes STRING, anio_mes STRING, dia_semana SMALLINT, es_fin_de_semana BOOLEAN""",
    'gold_dim_canal': 'canal_key INT, canal STRING',
    'gold_dim_categoria': 'categoria_key INT, categoria STRING',
    'gold_dim_cliente': """cliente_key INT, id_cliente STRING, id_cliente_maestro STRING,
        nombre STRING, ciudad STRING, canal_registro STRING, fecha_registro DATE,
        cohorte_registro DATE""",
    'gold_fact_pedido': """id_pedido STRING, cliente_key INT, fecha_key INT, canal_key INT,
        categoria_key INT, valor_bruto BIGINT, descuento BIGINT, venta_neta BIGINT, estado STRING,
        cuenta_como_venta BOOLEAN, fecha_entrega DATE, run_id STRING""",
    'gold_fact_pago': """id_pago STRING, id_pedido STRING, fecha_key INT, medio_pago STRING,
        valor BIGINT, es_reembolso BOOLEAN, run_id STRING""",
    'gold_fact_venta_neta_diaria': """fecha_key INT, canal_key INT, categoria_key INT,
        num_pedidos INT, venta_bruta BIGINT, descuento BIGINT, venta_neta BIGINT, run_id STRING""",
    'gold_fact_ticket': """id_ticket STRING, cliente_key INT, id_pedido STRING,
        fecha_creacion_key INT, canal_soporte STRING, estado STRING, tipo_caso STRING,
        tipo_caso_propuesto STRING, clasificacion_estado STRING, clasificacion_version STRING,
        horas_resolucion DOUBLE, run_id STRING""",
}

FACTS = ('gold_fact_pedido', 'gold_fact_pago', 'gold_fact_venta_neta_diaria', 'gold_fact_ticket')


# ------------------------------------------------------------- modelo estrella

@dataclass(frozen=True)
class Paso:
    """Un paso del modelo estrella: qué tabla, cómo se escribe y el SELECT que la calcula.

    El SELECT va separado de la escritura a propósito. Así la parte que decide los
    valores —la que tiene que dar exactamente lo mismo que DuckDB— se puede ejecutar
    y comparar sin tablas Delta ni permisos de escritura, y la parte que persiste se
    reduce a una línea por paso.
    """
    tabla: str
    modo: str           # 'anexar' (altas que no mueven claves) | 'reemplazar' (hechos)
    select: str


def sentencia(paso: Paso, T) -> str:
    """SQL de escritura del paso. `INSERT OVERWRITE` es atómico por tabla en Delta."""
    destino = T(paso.tabla)
    verbo = 'INSERT INTO' if paso.modo == 'anexar' else 'INSERT OVERWRITE TABLE'
    return f'{verbo} {destino} {paso.select}'


def _limpiar(sql: str) -> str:
    return ' '.join(sql.split())


def pasos_dimensiones(T, desde: date, hasta: date) -> list[Paso]:
    """Altas de dimensión, solo para valores nuevos: las claves ya emitidas no se mueven.

    `desde`/`hasta` se expanden a años completos igual que la versión local, porque la
    medida DAX de período comparable necesita el calendario sin huecos.
    """
    inicio, fin = date(desde.year, 1, 1), date(hasta.year, 12, 31)
    pasos = []
    for nombre in ('canal', 'categoria'):
        pasos.append(Paso(f'gold_dim_{nombre}', 'anexar', _limpiar(f"""
            SELECT (SELECT coalesce(max({nombre}_key), 0) FROM {T(f'gold_dim_{nombre}')})
                   + row_number() OVER (ORDER BY p.{nombre}) AS {nombre}_key, p.{nombre}
            FROM (SELECT DISTINCT {nombre} FROM {T('silver_pedidos')}) p
            WHERE NOT EXISTS (SELECT 1 FROM {T(f'gold_dim_{nombre}')} d WHERE d.{nombre} = p.{nombre})
        """)))
    pasos.append(Paso('gold_dim_cliente', 'anexar', _limpiar(f"""
        SELECT 0 AS cliente_key, 'DESCONOCIDO' AS id_cliente, 'DESCONOCIDO' AS id_cliente_maestro,
               CAST(NULL AS STRING) AS nombre, CAST(NULL AS STRING) AS ciudad,
               CAST(NULL AS STRING) AS canal_registro, CAST(NULL AS DATE) AS fecha_registro,
               CAST(NULL AS DATE) AS cohorte_registro
        WHERE NOT EXISTS (SELECT 1 FROM {T('gold_dim_cliente')} WHERE cliente_key = 0)
    """)))
    pasos.append(Paso('gold_dim_cliente', 'anexar', _limpiar(f"""
        SELECT (SELECT coalesce(max(cliente_key), 0) FROM {T('gold_dim_cliente')})
               + row_number() OVER (ORDER BY c.id_cliente) AS cliente_key,
               c.id_cliente, m.id_cliente_maestro, c.nombre, c.ciudad, c.canal_registro,
               c.fecha_registro, c.cohorte_registro
        FROM {T('silver_clientes')} c
        JOIN {T('v_cliente_maestro')} m ON m.id_cliente = c.id_cliente
        WHERE NOT EXISTS (SELECT 1 FROM {T('gold_dim_cliente')} d WHERE d.id_cliente = c.id_cliente)
    """)))
    meses = ', '.join(f"'{m}'" for m in MESES)
    pasos.append(Paso('gold_dim_fecha', 'anexar', _limpiar(f"""
        SELECT cast(date_format(g.d, 'yyyyMMdd') AS INT) AS fecha_key, g.d AS fecha,
               cast(year(g.d) AS SMALLINT) AS anio, cast(quarter(g.d) AS SMALLINT) AS trimestre,
               cast(month(g.d) AS SMALLINT) AS mes,
               element_at(array({meses}), month(g.d)) AS nombre_mes,
               date_format(g.d, 'yyyy-MM') AS anio_mes,
               cast(weekday(g.d) + 1 AS SMALLINT) AS dia_semana,
               weekday(g.d) + 1 >= 6 AS es_fin_de_semana
        FROM (SELECT explode(sequence(DATE'{inicio}', DATE'{fin}', interval 1 day)) AS d) g
        WHERE NOT EXISTS (SELECT 1 FROM {T('gold_dim_fecha')} f WHERE f.fecha = g.d)
    """)))
    return pasos


def sql_etiqueta_vigente(T) -> str:
    """Etiqueta vigente por texto: la última importada de cada estado, por separado.

    Espejo de la vista `v_etiqueta_vigente` de DuckDB. Mantener revisada y propuesta
    en columnas distintas es lo que impide que una propuesta se publique como validada.
    """
    return _limpiar(f"""
        WITH ordenadas AS (
            SELECT *, row_number() OVER (PARTITION BY texto_hash, estado
                                         ORDER BY importado_en DESC, version DESC) AS n
            FROM {T('ticket_clasificacion')}
        )
        SELECT texto_hash,
               max(CASE WHEN estado = 'REVISADA'  THEN tipo    END) AS tipo_revisado,
               max(CASE WHEN estado = 'REVISADA'  THEN version END) AS version_revisada,
               max(CASE WHEN estado = 'PROPUESTA' THEN tipo    END) AS tipo_propuesto,
               max(CASE WHEN estado = 'PROPUESTA' THEN version END) AS version_propuesta
        FROM ordenadas WHERE n = 1 GROUP BY texto_hash
    """)


def sql_cliente_maestro(T) -> str:
    """Mismo documento bajo dos ids es la misma persona. Vista, no columna: así refleja
    siempre el estado actual de Silver."""
    return _limpiar(f"""
        SELECT id_cliente,
               min(id_cliente) OVER (PARTITION BY coalesce(documento_limpio, id_cliente))
                   AS id_cliente_maestro
        FROM {T('silver_clientes')}
    """)


def pasos_hechos(T, run_id: str) -> list[Paso]:
    """Reconstrucción completa de los hechos, marcados con el run que los produjo.

    La reconstrucción completa prioriza auditabilidad para el tamaño del caso; no se
    presenta como carga incremental de hechos a escala.
    """
    return [
        Paso('gold_fact_pedido', 'reemplazar', _limpiar(f"""
            SELECT p.id_pedido,
                   CASE WHEN s.id_cliente IS NULL THEN 0 ELSE c.cliente_key END AS cliente_key,
                   f.fecha_key, can.canal_key, cat.categoria_key, p.valor_bruto, p.descuento,
                   CASE WHEN p.estado = 'ENTREGADO' THEN p.valor_neto ELSE 0 END AS venta_neta,
                   p.estado, p.estado = 'ENTREGADO' AS cuenta_como_venta, p.fecha_entrega,
                   '{run_id}' AS run_id
            FROM {T('silver_pedidos')} p
            LEFT JOIN {T('silver_clientes')} s ON s.id_cliente = p.id_cliente
            LEFT JOIN {T('gold_dim_cliente')} c ON c.id_cliente = s.id_cliente
            JOIN {T('gold_dim_fecha')} f ON f.fecha = p.fecha_pedido_dia
            JOIN {T('gold_dim_canal')} can ON can.canal = p.canal
            JOIN {T('gold_dim_categoria')} cat ON cat.categoria = p.categoria
        """)),
        Paso('gold_fact_pago', 'reemplazar', _limpiar(f"""
            SELECT g.id_pago, g.id_pedido, f.fecha_key, g.medio_pago, g.valor, g.es_reembolso,
                   '{run_id}' AS run_id
            FROM {T('silver_pagos')} g
            JOIN {T('gold_dim_fecha')} f ON f.fecha = g.fecha_pago
            JOIN {T('gold_fact_pedido')} o ON o.id_pedido = g.id_pedido
        """)),
        Paso('gold_fact_venta_neta_diaria', 'reemplazar', _limpiar(f"""
            SELECT f.fecha_key, c.canal_key, k.categoria_key, v.num_pedidos, v.venta_bruta,
                   v.descuento, v.venta_neta, '{run_id}' AS run_id
            FROM {T('gold_venta_neta_diaria')} v
            JOIN {T('gold_dim_fecha')} f ON f.fecha = v.fecha
            JOIN {T('gold_dim_canal')} c ON c.canal = v.canal
            JOIN {T('gold_dim_categoria')} k ON k.categoria = v.categoria
        """)),
        Paso('gold_fact_ticket', 'reemplazar', _limpiar(f"""
            SELECT t.id_ticket,
                   CASE WHEN s.id_cliente IS NULL THEN 0 ELSE c.cliente_key END AS cliente_key,
                   t.id_pedido, f.fecha_key AS fecha_creacion_key, t.canal_soporte, t.estado,
                   e.tipo_revisado AS tipo_caso, e.tipo_propuesto AS tipo_caso_propuesto,
                   CASE WHEN e.tipo_revisado IS NOT NULL THEN 'REVISADA'
                        WHEN e.tipo_propuesto IS NOT NULL THEN 'PROPUESTA' END
                        AS clasificacion_estado,
                   coalesce(e.version_revisada, e.version_propuesta) AS clasificacion_version,
                   t.horas_resolucion, '{run_id}' AS run_id
            FROM {T('silver_tickets')} t
            LEFT JOIN {T('silver_clientes')} s ON s.id_cliente = t.id_cliente
            LEFT JOIN {T('gold_dim_cliente')} c ON c.id_cliente = s.id_cliente
            LEFT JOIN {T('v_etiqueta_vigente')} e ON e.texto_hash = t.texto_hash
            JOIN {T('gold_dim_fecha')} f ON f.fecha = cast(t.fecha_creacion AS DATE)
        """)),
    ]


def comprobar_estrella(spark, T) -> dict:
    """La estrella no puede perder filas ni dejar de conciliar con el agregado diario.

    Es la misma comprobación que hace la versión local antes de confirmar la
    transacción; aquí corre antes de mover el puntero de publicación.
    """
    conteos = {}
    for fact, origen in [('gold_fact_pedido', 'silver_pedidos'), ('gold_fact_pago', 'silver_pagos'),
                         ('gold_fact_ticket', 'silver_tickets'),
                         ('gold_fact_venta_neta_diaria', 'gold_venta_neta_diaria')]:
        n = spark.sql(f'SELECT count(*) FROM {T(fact)}').first()[0]
        if n != spark.sql(f'SELECT count(*) FROM {T(origen)}').first()[0]:
            raise RuntimeError(f'Modelo estrella pierde filas: {fact}')
        conteos[fact] = n
    total, nulos = spark.sql(f"""SELECT coalesce(sum(venta_neta), 0),
        count(*) FILTER (WHERE cliente_key IS NULL) FROM {T('gold_fact_pedido')}""").first()
    diario = spark.sql(
        f"SELECT coalesce(sum(venta_neta), 0) FROM {T('gold_venta_neta_diaria')}").first()[0]
    if nulos or total != diario:
        raise RuntimeError('Modelo estrella no concilia')
    return conteos


def delta_refrescar_atributos_cliente(T) -> str:
    """Atributos descriptivos de `dim_cliente` al día, conservando la clave.

    Requiere Delta (`MERGE INTO`). **No ejecutado**: se valida en E04.
    """
    return ' '.join(f"""
        MERGE INTO {T('gold_dim_cliente')} d
        USING (SELECT c.id_cliente, m.id_cliente_maestro, c.nombre, c.ciudad, c.canal_registro,
                      c.fecha_registro, c.cohorte_registro
               FROM {T('silver_clientes')} c
               JOIN {T('v_cliente_maestro')} m ON m.id_cliente = c.id_cliente) s
        ON d.id_cliente = s.id_cliente
        WHEN MATCHED THEN UPDATE SET
            d.id_cliente_maestro = s.id_cliente_maestro, d.nombre = s.nombre, d.ciudad = s.ciudad,
            d.canal_registro = s.canal_registro, d.fecha_registro = s.fecha_registro,
            d.cohorte_registro = s.cohorte_registro
    """.split())


# ----------------------------------------------------- incidencias de calidad

def delta_sentencias_incidencias(T, tabla: str, run_id: str) -> list[str]:
    """Abre y resuelve incidencias sin borrar historia.

    Solo se resuelven los ids efectivamente reevaluados en esta corrida: la ausencia
    de una fila en el lote no es prueba de que el problema se corrigiera. Las vistas
    `ids_revisados` y `q_nueva` las registra el notebook antes de llamar.
    Requiere Delta (`MERGE INTO`). **No ejecutado**: se valida en E04.
    """
    return [' '.join(s.split()) for s in [f"""
        MERGE INTO {T('calidad_incidencias')} t
        USING (SELECT DISTINCT trim(id_registro) AS id_registro FROM ids_revisados) s
        ON t.tabla = '{tabla}' AND t.id_registro = s.id_registro
        WHEN MATCHED THEN UPDATE SET t.estado = 'RESUELTA', t.ultima_revision = current_timestamp(),
            t.run_id = '{run_id}', t.motivo = 'No incumple en la última reevaluación'
    """, f"""
        MERGE INTO {T('calidad_incidencias')} t
        USING (SELECT DISTINCT trim(id_registro) AS id_registro, regla_violada FROM q_nueva) s
        ON t.tabla = '{tabla}' AND t.id_registro = s.id_registro AND t.regla = s.regla_violada
        WHEN MATCHED THEN UPDATE SET t.estado = 'ABIERTA', t.ultima_revision = current_timestamp(),
            t.run_id = '{run_id}', t.motivo = 'Regla incumplida'
        WHEN NOT MATCHED THEN INSERT (tabla, id_registro, regla, estado, primera_observacion,
            ultima_revision, run_id, motivo)
        VALUES ('{tabla}', s.id_registro, s.regla_violada, 'ABIERTA', current_timestamp(),
            current_timestamp(), '{run_id}', 'Regla incumplida')
    """]]


# ------------------------------------------------------ publicación consistente

@dataclass(frozen=True)
class PoliticaPublicacion:
    """Sin umbrales inventados por defecto; configurar tras acordarlos con negocio (U08)."""
    max_fraccion_cuarentena: float | None = None
    min_pedidos: int | None = None
    venta_esperada: int | None = None
    tolerancia: int = 0
    fecha_minima_datos: date | None = None

    def __post_init__(self):
        if self.max_fraccion_cuarentena is not None and not 0 <= self.max_fraccion_cuarentena <= 1:
            raise ValueError('Fracción de cuarentena debe estar entre 0 y 1')
        if self.tolerancia < 0 or (self.min_pedidos is not None and self.min_pedidos < 0):
            raise ValueError('Tolerancia y mínimo no pueden ser negativos')


class PublicacionBloqueada(RuntimeError):
    pass


def sql_conciliacion(T) -> str:
    """Filas en las que Gold y Silver no dicen lo mismo, al mismo grano.

    `EXCEPT ALL` en los dos sentidos detecta tanto lo que falta en Gold como lo que
    sobra, que es justo el daño que deja una carga interrumpida entre tablas.
    """
    return ' '.join(f"""
        WITH esperado AS (
            SELECT fecha_pedido_dia AS fecha, canal, categoria, count(*) AS num_pedidos,
                   sum(valor_bruto) AS venta_bruta, sum(descuento) AS descuento,
                   sum(valor_neto) AS venta_neta
            FROM {T('silver_pedidos')} WHERE estado = 'ENTREGADO' GROUP BY 1, 2, 3
        ), publicado AS (
            SELECT fecha, canal, categoria, num_pedidos, venta_bruta, descuento, venta_neta
            FROM {T('gold_venta_neta_diaria')}
        ), diferencias AS (
            (SELECT * FROM esperado EXCEPT ALL SELECT * FROM publicado)
            UNION ALL
            (SELECT * FROM publicado EXCEPT ALL SELECT * FROM esperado)
        ) SELECT count(*) FROM diferencias
    """.split())


def validar_publicacion(spark, T, resumen: dict | None = None,
                        politica: PoliticaPublicacion | None = None) -> dict:
    """Compuertas antes de mover el puntero de publicación. Espejo de `operations.validar_publicacion`."""
    politica = politica or PoliticaPublicacion()
    resumen = resumen or {}
    n, venta, fecha = spark.sql(f"""
        SELECT count(*), coalesce(sum(CASE WHEN estado = 'ENTREGADO' THEN valor_neto ELSE 0 END), 0),
               max(fecha_pedido_dia) FROM {T('silver_pedidos')}""").first()
    gold = spark.sql(
        f"SELECT coalesce(sum(venta_neta), 0) FROM {T('gold_venta_neta_diaria')}").first()[0]
    diferencias = spark.sql(sql_conciliacion(T)).first()[0]

    errores = []
    if venta != gold or diferencias:
        errores.append('Silver y Gold no concilian')
    if politica.min_pedidos is not None and n < politica.min_pedidos:
        errores.append('Volumen menor que mínimo configurado')
    if politica.venta_esperada is not None and abs(venta - politica.venta_esperada) > politica.tolerancia:
        errores.append('Venta fuera de tolerancia configurada')
    if politica.fecha_minima_datos and (fecha is None or fecha < politica.fecha_minima_datos):
        errores.append('Frescura insuficiente')
    for tabla in ('clientes', 'pedidos', 'pagos', 'tickets_soporte'):
        x = resumen.get(tabla, {})
        if politica.max_fraccion_cuarentena is not None and x.get('leidos', 0):
            if x.get('rechazados_lote', 0) / x['leidos'] > politica.max_fraccion_cuarentena:
                errores.append(f'{tabla}: cuarentena supera umbral configurado')
    if errores:
        raise PublicacionBloqueada('; '.join(errores))
    return {'estado': 'VALIDADO', 'pedidos': n, 'venta_neta': int(venta),
            'fecha_maxima': fecha.isoformat() if fecha else None,
            'politica': 'configurada' if politica != PoliticaPublicacion() else 'conciliacion_basica'}


def sentencias_vistas_publicadas(T, temporal: bool = False) -> list[str]:
    """Vistas de consumo: solo el run publicado.

    Es lo que sustituye a la transacción de DuckDB. Mientras `publicacion_actual` no
    apunte al run nuevo, Power BI y el reporte siguen viendo el anterior completo,
    aunque las tablas de hechos ya tengan filas a medio escribir.

    `temporal` existe solo para las pruebas: ahí las tablas son vistas temporales, y
    Spark no deja que una vista persistente referencie una temporal. El cuerpo de la
    consulta es el mismo en los dos casos.
    """
    clase = 'TEMP VIEW' if temporal else 'VIEW'
    return [' '.join(f"""
        CREATE OR REPLACE {clase} {T('v_' + fact.replace('gold_', ''))} AS
        SELECT * FROM {T(fact)}
        WHERE run_id = (SELECT run_id FROM {T('publicacion_actual')} WHERE id = 1)
    """.split()) for fact in FACTS]


def delta_publicar(T, run_id: str, venta_neta: int) -> str:
    """Movimiento del puntero de publicación: una sola escritura, atómica en Delta.

    Requiere Delta (`MERGE INTO`). **No ejecutado**: se valida en E04.
    """
    return ' '.join(f"""
        MERGE INTO {T('publicacion_actual')} t
        USING (SELECT 1 AS id, '{run_id}' AS run_id, current_timestamp() AS validado_en,
                      CAST({venta_neta} AS BIGINT) AS venta_neta) s
        ON t.id = s.id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """.split())


# ------------------------------------------------- contratos de archivo entrada

class ErrorContrato(ValueError):
    pass


CONTRATOS = {
    'clientes': ['id_cliente', 'nombre', 'documento', 'ciudad', 'canal_registro', 'fecha_registro'],
    'pedidos': ['id_pedido', 'id_cliente', 'fecha_pedido', 'canal', 'categoria',
                'valor_bruto', 'descuento', 'estado', 'fecha_entrega'],
    'pagos': ['id_pago', 'id_pedido', 'fecha_pago', 'valor', 'medio_pago', 'fecha_carga'],
    'tickets_soporte': COLS_TICKETS,
}


def verificar_contrato(spark, ruta: str, tabla: str) -> dict:
    """Contrato estructural del CSV antes de leerlo como datos.

    Un archivo inválido bloquea la corrida entera, no sus filas: si la cabecera
    cambió, lo que haya debajo no significa lo que el proceso cree.

    Diferencia declarada con la versión local: allí se valida la codificación byte a
    byte con el lector de Python. Aquí Spark sustituye los bytes inválidos por el
    carácter de reemplazo, así que esa comprobación se hace sobre su presencia, no
    sobre una excepción del decodificador.
    """
    esperadas = CONTRATOS[tabla]
    cabecera = (spark.read.option('header', False).option('encoding', 'UTF-8')
                .csv(ruta).limit(1).first())
    if cabecera is None:
        raise ErrorContrato(f'{tabla}: archivo sin cabecera')
    nombres = [(c or '').strip().lstrip('﻿') for c in cabecera]
    if not nombres or len(nombres) != len(set(nombres)):
        raise ErrorContrato(f'{tabla}: cabecera vacía o repetida')
    faltan = sorted(set(esperadas) - set(nombres))
    if faltan:
        raise ErrorContrato(f'{tabla}: faltan columnas {faltan}')

    esquema = ', '.join(f'`{c}` string' for c in nombres) + ', _corrupto string'
    df = (spark.read.option('header', True).option('mode', 'PERMISSIVE')
          .option('columnNameOfCorruptRecord', '_corrupto').option('encoding', 'UTF-8')
          .schema(esquema).csv(ruta)).cache()
    malformadas = df.filter(F.col('_corrupto').isNotNull()).count()
    if malformadas:
        raise ErrorContrato(f'{tabla}: {malformadas} registros con ancho inválido')
    sucias = df.filter(reduce_or([F.col(f'`{c}`').contains('�') for c in nombres])).count()
    if sucias:
        raise ErrorContrato(f'{tabla}: {sucias} registros con codificación inválida')
    filas = df.count()
    df.unpersist()
    return {'filas': filas, 'extras': sorted(set(nombres) - set(esperadas))}


def reduce_or(columnas):
    from functools import reduce
    return reduce(lambda a, b: a | b, columnas)


In [ ]:
# === 4. Tablas Delta (se crean una sola vez) ===
ddl = {
    "silver_clientes": """id_cliente STRING, nombre STRING, documento_original STRING, documento_limpio STRING,
        ciudad STRING, canal_registro STRING, fecha_registro DATE, cohorte_registro DATE,
        row_hash STRING, ingested_at TIMESTAMP""",
    "silver_pedidos": """id_pedido STRING, id_cliente STRING, fecha_pedido TIMESTAMP, fecha_pedido_dia DATE,
        canal STRING, categoria STRING, valor_bruto BIGINT, descuento BIGINT, valor_neto BIGINT,
        estado STRING, fecha_entrega DATE, flag_cliente_desconocido BOOLEAN, hash_negocio STRING,
        row_hash STRING, ingested_at TIMESTAMP""",
    "silver_pagos": """id_pago STRING, id_pedido STRING, fecha_pago DATE, valor BIGINT, es_reembolso BOOLEAN,
        medio_pago STRING, fecha_carga DATE, hash_negocio STRING, row_hash STRING, ingested_at TIMESTAMP""",
    "cuarentena": """tabla_origen STRING, id_registro STRING, regla_violada STRING, row_hash STRING,
        registro_json STRING, run_id STRING, fecha_proceso TIMESTAMP""",
    "etl_watermark": "tabla STRING, ultimo_valor DATE, ultimo_run_id STRING, actualizado_en TIMESTAMP",
    "bronze_manifest": "tabla STRING, sha256 STRING, ruta STRING, filas BIGINT, run_id STRING, ingested_at TIMESTAMP",
    "gold_venta_neta_diaria": """fecha DATE, canal STRING, categoria STRING, num_pedidos BIGINT,
        venta_bruta BIGINT, descuento BIGINT, venta_neta BIGINT""",
}
ddl.update(DDL_EXTENSIONES)          # soporte, estrella, incidencias y publicación
for tabla, columnas in ddl.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {T(tabla)} ({columnas}) USING DELTA")
spark.sql(f"CREATE OR REPLACE VIEW {T('v_cliente_maestro')} AS " + sql_cliente_maestro(T))
spark.sql(f"CREATE OR REPLACE VIEW {T('v_etiqueta_vigente')} AS " + sql_etiqueta_vigente(T))


In [ ]:
# === 5. Contrato de archivo: un CSV inválido detiene la corrida, no sus filas ===
# Si la cabecera cambió, lo que viene debajo no significa lo que el proceso cree.
# Por eso esto corre antes de aterrizar nada en Bronze.
contratos = {}
for _tabla in ("clientes", "pedidos", "pagos", "tickets_soporte"):
    try:
        contratos[_tabla] = verificar_contrato(spark, f"{RAW}/{_tabla}.csv", _tabla)
    except AnalysisException:
        if _tabla == "tickets_soporte":      # soporte es opcional
            contratos[_tabla] = None
            continue
        raise
for _tabla, _info in contratos.items():
    print(f"{_tabla:16}", "ausente (opcional)" if _info is None else
          f"{_info['filas']:>7} filas, columnas extra: {_info['extras'] or 'ninguna'}")


In [ ]:
# === 6. Bronze: copia cruda en Parquet, inmutable e idempotente ===
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

SCHEMA_MANIFEST = StructType([
    StructField("tabla", StringType()), StructField("sha256", StringType()), StructField("ruta", StringType()),
    StructField("filas", LongType()), StructField("run_id", StringType()), StructField("ingested_at", TimestampType())])


def aterrizar(tabla, run_id):
    """Aterriza <tabla>.csv en Bronze si su contenido (sha256) es nuevo; devuelve lo que hay en Bronze.
    El mismo archivo dos veces no escribe nada; uno distinto crea un snapshot nuevo y no toca el anterior."""
    origen = f"{RAW}/{tabla}.csv"
    sha = spark.read.format("binaryFile").load(origen).select(F.sha2("content", 256)).first()[0]
    previo = (spark.table(T("bronze_manifest"))
              .filter((F.col("tabla") == tabla) & (F.col("sha256") == sha)).select("ruta").collect())
    if previo:
        ruta, nuevo = previo[0][0], False
    else:
        # todo como texto: tipar y validar es trabajo de Silver, no de Bronze
        df = spark.read.csv(origen, header=True, inferSchema=False)
        ahora = datetime.now()
        ruta = f"{BRONZE}/{tabla}/ingest_date={ahora:%Y-%m-%d}/{tabla}_{sha[:12]}"
        (df.withColumn("_ingested_at", F.lit(ahora)).withColumn("_run_id", F.lit(run_id))
           .withColumn("_source_file", F.lit(f"{tabla}.csv")).withColumn("_source_sha256", F.lit(sha))
           .write.mode("errorifexists").parquet(ruta))
        filas = spark.read.parquet(ruta).count()
        (spark.createDataFrame([(tabla, sha, ruta, filas, run_id, ahora)], SCHEMA_MANIFEST)
              .write.mode("append").saveAsTable(T("bronze_manifest")))
        nuevo = True
    leido = spark.read.parquet(ruta)
    leido = leido.select(*[c for c in leido.columns if not c.startswith("_")])
    return leido, nuevo


In [ ]:
# === 7. Clientes -> pedidos -> pagos (cada paso es idempotente por sí mismo) ===
run_id, inicio = str(uuid.uuid4()), datetime.now()
resumen = {}


def metricas_ultimo_merge(tabla):
    m = spark.sql(f"DESCRIBE HISTORY {T(tabla)} LIMIT 1").select("operationMetrics").first()[0]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))


def merge_silver(df, tabla, clave):
    """Inserta lo nuevo y reemplaza lo que cambió de row_hash; lo idéntico no se toca."""
    df.withColumn("ingested_at", F.current_timestamp()).createOrReplaceTempView("stg")
    spark.sql(f"""
        MERGE INTO {T(tabla)} t USING stg s ON t.{clave} = s.{clave}
        WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *""")
    return metricas_ultimo_merge(tabla)


def merge_cuarentena(df):
    """Una fila por (tabla, registro, regla, contenido). Reprocesar el mismo archivo no duplica filas."""
    (df.dropDuplicates(["tabla_origen", "id_registro", "regla_violada", "row_hash"])
       .withColumn("run_id", F.lit(run_id)).withColumn("fecha_proceso", F.current_timestamp())
       .createOrReplaceTempView("stg_q"))
    spark.sql(f"""
        MERGE INTO {T("cuarentena")} t USING stg_q s
        ON t.tabla_origen = s.tabla_origen AND t.id_registro = s.id_registro
           AND t.regla_violada = s.regla_violada AND t.row_hash = s.row_hash
        WHEN NOT MATCHED THEN INSERT *""")
    return int(spark.sql(f"DESCRIBE HISTORY {T('cuarentena')} LIMIT 1")
               .select("operationMetrics").first()[0].get("numTargetRowsInserted", 0))


# ---- clientes
cli_raw, nuevo_b = aterrizar("clientes", run_id)
cli_ok, cli_q = validar_clientes(cli_raw)
ins, upd = merge_silver(cli_ok, "silver_clientes", "id_cliente")
resumen["clientes"] = dict(insertados=ins, actualizados=upd, cuarentena_nueva=merge_cuarentena(cli_q), bronze_nuevo=nuevo_b)

# ---- pedidos: la fuente no trae fecha de carga, así que el incremental es por contenido (row_hash)
ped_raw, nuevo_b = aterrizar("pedidos", run_id)
negocio = (spark.table(T("silver_pedidos")).filter(F.col("hash_negocio").isNotNull())
           .groupBy("hash_negocio").agg(F.min("id_pedido").alias("id_pedido")))
ped_ok, ped_q = validar_pedidos(ped_raw, FECHA_MIN, FECHA_MAX, spark.table(T("silver_clientes")).select("id_cliente"), negocio)
ped_ok = ped_ok.cache()

# Días de venta a recalcular: el día nuevo de cada fila escrita y el día anterior si ya existía
previo = spark.table(T("silver_pedidos")).select("id_pedido", F.col("row_hash").alias("h_prev"), F.col("fecha_pedido_dia").alias("dia_prev"))
cambios = ped_ok.join(previo, "id_pedido", "left").filter(F.col("h_prev").isNull() | (F.col("h_prev") != F.col("row_hash")))
dias = sorted({r[0] for r in cambios.select("fecha_pedido_dia").union(cambios.select("dia_prev")).distinct().collect() if r[0]})


def rechazados(q, ok, clave):
    """Ids que fueron a cuarentena en esta corrida y no quedaron entre los válidos. Se recogen
    antes del MERGE: Spark es perezoso y, si no, se recalcularía contra el Silver ya modificado."""
    ids = q.select(F.trim("id_registro").alias(clave)).filter(F.col(clave) != "").distinct()
    return [r[0] for r in ids.join(ok.select(clave), clave, "left_anti").collect()]


def retirar(tabla, clave, ids):
    """Borra de Silver los registros que hoy son inválidos (ya quedaron en cuarentena)."""
    if not ids:
        return 0
    spark.createDataFrame([(i,) for i in ids], f"{clave} string").createOrReplaceTempView("ids_retiro")
    spark.sql(f"DELETE FROM {T(tabla)} WHERE {clave} IN (SELECT {clave} FROM ids_retiro)")
    return int(spark.sql(f"DESCRIBE HISTORY {T(tabla)} LIMIT 1")
               .select("operationMetrics").first()[0].get("numDeletedRows", 0))


ped_rechazados = rechazados(ped_q, ped_ok, "id_pedido")
ins, upd = merge_silver(ped_ok, "silver_pedidos", "id_pedido")
resumen["pedidos"] = dict(insertados=ins, actualizados=upd, cuarentena_nueva=merge_cuarentena(ped_q), bronze_nuevo=nuevo_b)
# Un pedido que ya estaba en Silver y hoy es inválido sale de Silver y deja de sumar venta.
dias_retirados = (spark.table(T("silver_pedidos")).filter(F.col("id_pedido").isin(ped_rechazados))
                  .select("fecha_pedido_dia").distinct().collect())
dias = sorted(set(dias) | {r[0] for r in dias_retirados})
resumen["pedidos"]["retirados"] = retirar("silver_pedidos", "id_pedido", ped_rechazados)

# ---- pagos: incremental por watermark de fecha_carga con ventana de reproceso
pag_raw, nuevo_b = aterrizar("pagos", run_id)
wm = spark.table(T("etl_watermark")).filter(F.col("tabla") == "pagos").select("ultimo_valor").first()
carga = F.expr("try_to_timestamp(fecha_carga, 'yyyy-MM-dd')")
if wm:
    # Los pagos que esperan a su pedido se releen siempre, aunque estén fuera de la ventana.
    pendientes = [r[0] for r in (
        spark.table(T("cuarentena"))
        .filter((F.col("tabla_origen") == "pagos") & F.col("regla_violada").isin("PAGO_HUERFANO", "PEDIDO_EN_CUARENTENA"))
        .select(F.trim("id_registro").alias("id_pago"))
        .join(spark.table(T("silver_pagos")), "id_pago", "left_anti")
        .union(spark.table(T("silver_pagos")).join(spark.table(T("silver_pedidos")), "id_pedido", "left_anti")
               .select("id_pago"))
        .distinct().collect())]
    pag_raw = pag_raw.filter(carga.isNull() | (carga >= F.expr(f"date_sub(DATE'{wm[0]}', {LOOKBACK_DIAS})"))
                             | F.trim("id_pago").isin(pendientes))
pag_raw = pag_raw.cache()
pedidos_dia = spark.table(T("silver_pedidos")).select("id_pedido", "fecha_pedido_dia")
ped_en_q = spark.table(T("cuarentena")).filter(F.col("tabla_origen") == "pedidos").select("id_registro").distinct()
negocio_pag = (spark.table(T("silver_pagos")).filter(F.col("hash_negocio").isNotNull())
               .groupBy("hash_negocio").agg(F.min("id_pago").alias("id_pago")))
pag_ok, pag_q = validar_pagos(pag_raw, pedidos_dia, ped_en_q, negocio_pag)
pag_ok = pag_ok.cache()
pag_rechazados = rechazados(pag_q, pag_ok, "id_pago")
ins, upd = merge_silver(pag_ok, "silver_pagos", "id_pago")
resumen["pagos"] = dict(insertados=ins, actualizados=upd, cuarentena_nueva=merge_cuarentena(pag_q), bronze_nuevo=nuevo_b)
resumen["pagos"]["retirados"] = retirar("silver_pagos", "id_pago", pag_rechazados)

maximo = pag_raw.select(F.max(carga).cast("date")).first()[0]
if maximo:
    nuevo_wm = max(maximo, wm[0]) if wm else maximo
    spark.sql(f"""
        MERGE INTO {T("etl_watermark")} t
        USING (SELECT 'pagos' AS tabla, DATE'{nuevo_wm}' AS ultimo_valor, '{run_id}' AS ultimo_run_id,
                      current_timestamp() AS actualizado_en) s
        ON t.tabla = s.tabla
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *""")


In [ ]:
# === 8. Soporte: opcional, y nunca altera el cálculo de venta ===
# Las fechas incoherentes se conservan marcadas y sin horas de resolución: inventar un
# SLA que la fuente no respalda sería peor que no tenerlo.
if contratos.get("tickets_soporte"):
    tic_raw, nuevo_b = aterrizar("tickets_soporte", run_id)
    tic_ok, tic_q = validar_tickets(tic_raw)
    tic_rechazados = rechazados(tic_q, tic_ok, "id_ticket")
    ins, upd = merge_silver(tic_ok, "silver_tickets", "id_ticket")
    resumen["tickets_soporte"] = dict(insertados=ins, actualizados=upd,
                                      cuarentena_nueva=merge_cuarentena(tic_q), bronze_nuevo=nuevo_b)
    resumen["tickets_soporte"]["retirados"] = retirar("silver_tickets", "id_ticket", tic_rechazados)


# ---- incidencias de calidad: se abren y se resuelven, nunca se borra el historial
def actualizar_incidencias(tabla, raw, q, col_id):
    """Solo se resuelven los ids efectivamente reevaluados en esta corrida. Que una fila
    no venga en el lote no prueba que su problema se haya corregido."""
    raw.select(F.trim(F.col(col_id)).alias("id_registro")).createOrReplaceTempView("ids_revisados")
    q.select("id_registro", "regla_violada").createOrReplaceTempView("q_nueva")
    for sentencia_sql in delta_sentencias_incidencias(T, tabla, run_id):
        spark.sql(sentencia_sql)


for _tabla, _raw, _q, _col in [("clientes", cli_raw, cli_q, "id_cliente"),
                               ("pedidos", ped_raw, ped_q, "id_pedido"),
                               ("pagos", pag_raw, pag_q, "id_pago")]:
    actualizar_incidencias(_tabla, _raw, _q, _col)
if contratos.get("tickets_soporte"):
    actualizar_incidencias("tickets_soporte", tic_raw, tic_q, "id_ticket")


In [ ]:
# === 9. Venta neta diaria: se recalcula solo para los días tocados (depende solo de Silver) ===
if RECALCULAR_GOLD:
    dias = sorted({r[0] for r in spark.table(T("silver_pedidos")).select("fecha_pedido_dia")
                   .union(spark.table(T("gold_venta_neta_diaria")).select("fecha")).distinct().collect()})
if dias:
    lista = ", ".join(f"DATE'{d}'" for d in dias)
    spark.sql(f"DELETE FROM {T('gold_venta_neta_diaria')} WHERE fecha IN ({lista})")
    spark.sql(f"""
        INSERT INTO {T('gold_venta_neta_diaria')}
        SELECT fecha_pedido_dia, canal, categoria, count(*), sum(valor_bruto), sum(descuento), sum(valor_neto)
        FROM {T('silver_pedidos')}
        WHERE estado = '{ESTADO_VENTA}' AND fecha_pedido_dia IN ({lista})
        GROUP BY fecha_pedido_dia, canal, categoria""")
resumen["dias_recalculados"] = len(dias)

for tabla in ("clientes", "pedidos", "pagos"):
    r = resumen[tabla]
    print(f"{tabla:9} insertados={r['insertados']:>6} actualizados={r['actualizados']:>4} "
          f"cuarentena_nueva={r['cuarentena_nueva']:>4} bronze={'nuevo' if r['bronze_nuevo'] else 'ya existia'}")
print("dias de venta recalculados:", resumen["dias_recalculados"])


In [ ]:
# === 10. Modelo estrella: dimensiones estables y hechos marcados con el run ===
# Las claves ya emitidas no se mueven entre cargas: si se renumeraran, cualquier informe
# guardado contra ellas quedaría apuntando a otra cosa.
from datetime import date as _date

_rango = spark.sql(f"""
    SELECT least(coalesce(min(f), DATE'{FECHA_MIN}'), DATE'{FECHA_MIN}'),
           greatest(coalesce(max(f), DATE'{FECHA_MAX}'), DATE'{FECHA_MAX}')
    FROM (SELECT fecha_pedido_dia AS f FROM {T('silver_pedidos')}
          UNION SELECT fecha_pago FROM {T('silver_pagos')}
          UNION SELECT cast(fecha_creacion AS DATE) FROM {T('silver_tickets')})""").first()

for paso in pasos_dimensiones(T, _rango[0], _rango[1]):
    spark.sql(sentencia(paso, T))
spark.sql(delta_refrescar_atributos_cliente(T))      # atributos al día, misma clave
for paso in pasos_hechos(T, run_id):
    spark.sql(sentencia(paso, T))
resumen["estrella"] = comprobar_estrella(spark, T)
print("estrella:", resumen["estrella"])


In [ ]:
# === 11. Publicación consistente sin transacción entre tablas ===
# DuckDB envuelve toda la corrida en una transacción; Delta solo garantiza atomicidad por
# tabla. Aquí la consistencia se consigue al revés: los hechos ya están escritos con su
# run_id, pero el consumidor lee las vistas v_fact_*, que filtran por el run publicado.
# Mientras el puntero no se mueva, Power BI y el reporte siguen viendo la carga anterior
# completa, nunca una a medias.
POLITICA = PoliticaPublicacion()      # sin umbrales inventados; configurar tras acordar U08

for sentencia_sql in sentencias_vistas_publicadas(T):
    spark.sql(sentencia_sql)

try:
    resumen["publicacion"] = validar_publicacion(spark, T, resumen, POLITICA)
except PublicacionBloqueada as e:
    # La carga queda escrita pero NO publicada: el puntero sigue en el run anterior.
    print("PUBLICACIÓN BLOQUEADA:", e)
    raise
spark.sql(delta_publicar(T, run_id, resumen["publicacion"]["venta_neta"]))
print("publicado:", resumen["publicacion"])


In [ ]:
# === 12. Verificación contra el ETL local (solo aplica con el conjunto de datos completo del caso) ===
ESPERADO = {"pedidos_validos": 19270, "pagos_validos": 20124, "clientes_validos": 6040,
            "filas_gold": 7925, "venta_neta": 14_177_333_600}
real = {
    "pedidos_validos": spark.table(T("silver_pedidos")).count(),
    "pagos_validos": spark.table(T("silver_pagos")).count(),
    "clientes_validos": spark.table(T("silver_clientes")).count(),
    "filas_gold": spark.table(T("gold_venta_neta_diaria")).count(),
    "venta_neta": spark.table(T("gold_venta_neta_diaria")).agg(F.sum("venta_neta")).first()[0],
}
for k, v in ESPERADO.items():
    print(f"{k:18} esperado={v:>16,} real={real[k]:>16,} {'OK' if real[k] == v else 'DIFERENTE'}")
assert real == ESPERADO, "Las cifras no coinciden con el ETL local"

display(spark.table(T("cuarentena")).groupBy("tabla_origen", "regla_violada")
        .agg(F.countDistinct("id_registro").alias("registros")).orderBy("tabla_origen", F.desc("registros")))
